# ARG Constraints for Ancestral Recombination Graphs

This notebook demonstrates how to use ARG constraints to enforce that specific ancestral haplotypes must exist in an ancestral recombination graph. Unlike tree constraints (which apply to a single locus), ARG constraints specify per-locus requirements for a single ancestral haplotype that may have different descendants at different loci due to recombination.

In [ ]:
import numpy as np
from arg_constraint import ARGConstraint, ARGConstraintSet, ARGLineage

## 1. ARG vs Tree: The Key Difference

In a **tree**, all loci share the same genealogy. In an **ARG**, recombination causes different loci to have different genealogies.

An ancestral haplotype (node in the ARG) can have:
- Different descendants at different loci
- The same descendants at all loci (if no recombination affected it)

In [ ]:
# Example: 5 samples, 2 loci
# An ancestral haplotype that is ancestral to:
#   - samples {1, 2, 3} at locus 0
#   - samples {2, 3} at locus 1
# This could happen if sample 1 recombined away from {2,3} between the two loci

mask = np.array([
    [0, 0],  # sample 0: not in clade at either locus
    [1, 0],  # sample 1: in clade at locus 0 only
    [1, 1],  # sample 2: in clade at both loci
    [1, 1],  # sample 3: in clade at both loci
    [0, 0],  # sample 4: not in clade at either locus
])

print("Constraint mask (rows=samples, cols=loci):")
print(mask)
print()
print("Interpretation:")
print("  Locus 0: samples {1, 2, 3} must form a clade")
print("  Locus 1: samples {2, 3} must form a clade")
print("  Both clades must be satisfied by the SAME ancestral haplotype")

## 2. Creating ARG Constraints

There are two ways to create an `ARGConstraint`:

In [ ]:
# Method 1: From a 2D numpy array
constraint1 = ARGConstraint(mask)
print(f"From array: {constraint1}")
print(f"  Indices at locus 0: {constraint1.indices_at_locus(0)}")
print(f"  Indices at locus 1: {constraint1.indices_at_locus(1)}")
print()

# Method 2: From a list of index sets (one per locus)
constraint2 = ARGConstraint.from_indices(
    indices_per_locus=[{0, 1}, {1, 2}],  # locus 0: {0,1}, locus 1: {1,2}
    n_samples=5
)
print(f"From indices: {constraint2}")

## 3. ARG Lineages

An `ARGLineage` tracks which samples are descendants at each locus. Initially, each sample is its own lineage (ancestral to itself at all loci).

In [ ]:
n_samples, n_loci = 5, 2

# Create initial lineages
lin = {i: ARGLineage.from_sample(i, n_samples, n_loci) for i in range(n_samples)}

print("Initial lineages:")
for i in range(n_samples):
    print(f"  Sample {i}: {lin[i]}")

print()
print(f"Lineage 2 descendants at locus 0: {lin[2].descendants_at_locus(0)}")
print(f"Lineage 2 descendants at locus 1: {lin[2].descendants_at_locus(1)}")

## 4. Coalescence in an ARG

Lineages can coalesce at all loci (standard coalescence) or at specific loci only (representing recombination).

In [ ]:
# Coalescence at all loci (no recombination)
merged_23 = ARGLineage.coalesce(lin[2], lin[3])
print(f"Merge 2+3 at all loci: {merged_23}")
print(f"  Descendants at locus 0: {merged_23.descendants_at_locus(0)}")
print(f"  Descendants at locus 1: {merged_23.descendants_at_locus(1)}")
print()

# Coalescence at specific loci only (recombination)
partial = ARGLineage.coalesce(lin[0], lin[1], loci={0})  # Only at locus 0
print(f"Merge 0+1 at locus 0 only: {partial}")
print(f"  Descendants at locus 0: {partial.descendants_at_locus(0)}")
print(f"  Descendants at locus 1: {partial.descendants_at_locus(1)}")

## 5. Coalescence Rules Under Constraints

An `ARGConstraint` restricts which lineages can coalesce at each locus:

- **Neither in clade**: Can coalesce freely at that locus
- **Both entirely in clade**: Can coalesce (building the required clade)
- **One covers entire clade**: Clade is satisfied at that locus, can coalesce freely
- **Mixed (one in, one out)**: Cannot coalesce at that locus

In [ ]:
# Constraint: {1,2,3} at locus 0, {2,3} at locus 1
constraint = ARGConstraint(mask)
print(f"Constraint: {constraint}")
print()

# Reset lineages
lin = {i: ARGLineage.from_sample(i, n_samples, n_loci) for i in range(n_samples)}

print("Can coalesce (initially)?")
print(f"  2+3 (both in clade at both loci):    {constraint.can_coalesce(lin[2], lin[3])}")
print(f"  0+4 (neither in clade):              {constraint.can_coalesce(lin[0], lin[4])}")
print(f"  0+2 (0 outside, 2 inside):           {constraint.can_coalesce(lin[0], lin[2])}")
print(f"  1+2 (both in clade at locus 0):      {constraint.can_coalesce(lin[1], lin[2])}")

Note that 1+2 cannot coalesce initially because:
- At locus 0: both are in the clade {1,2,3} ✓
- At locus 1: sample 1 is **outside** the clade {2,3}, sample 2 is **inside** ✗

Sample 2 must first coalesce with sample 3 (completing the clade at locus 1) before it can coalesce with sample 1.

## 6. Constraint Satisfaction

A constraint is satisfied when a **single lineage** covers all required samples at **all loci**. This represents the ancestral haplotype existing in the ARG.

In [ ]:
print("Tracking constraint satisfaction:")
print()

# Initial state
lineages = set(lin.values())
print(f"All separate: satisfied = {constraint.is_satisfied(lineages)}")

# After merging 2+3
merged_23 = ARGLineage.coalesce(lin[2], lin[3])
lineages = {lin[0], lin[1], merged_23, lin[4]}
print(f"After 2+3:   satisfied = {constraint.is_satisfied(lineages)}")
print(f"  (Locus 1 clade {2,3} is complete, but locus 0 clade {1,2,3} is not)")

# After merging 1 with (2,3)
merged_123 = ARGLineage.coalesce(lin[1], merged_23)
lineages = {lin[0], merged_123, lin[4]}
print(f"After 1+(2,3): satisfied = {constraint.is_satisfied(lineages)}")
print(f"  (Both clades complete in lineage {merged_123})")

## 7. Managing Multiple Constraints

`ARGConstraintSet` manages multiple constraints and validates that no sample appears in multiple constraints at the same locus.

In [ ]:
cs = ARGConstraintSet(n_loci=2)

# Add non-overlapping constraints
cs.add(ARGConstraint.from_indices([{0, 1}, {0}], n_samples=6))  # Haplotype A
cs.add(ARGConstraint.from_indices([{3, 4}, {3, 4, 5}], n_samples=6))  # Haplotype B

print(f"Constraint set: {cs}")
print("\nConstraints:")
for c in cs:
    print(f"  {c}")

### Overlap Validation

A sample cannot be in multiple constraints at the same locus (it can only have one ancestor at each locus):

In [ ]:
cs_test = ARGConstraintSet(n_loci=2)
cs_test.add(ARGConstraint.from_indices([{0, 1}, {0, 1}], n_samples=4))

try:
    # This overlaps at sample 1, locus 0
    cs_test.add(ARGConstraint.from_indices([{1, 2}, {2, 3}], n_samples=4))
    print("ERROR: Should have raised ValueError")
except ValueError as e:
    print(f"Correctly rejected: {e}")

Note: The same sample CAN appear in different constraints at **different loci**:

In [ ]:
cs_ok = ARGConstraintSet(n_loci=2)
cs_ok.add(ARGConstraint.from_indices([{0, 1}, set()], n_samples=4))  # Sample 1 at locus 0
cs_ok.add(ARGConstraint.from_indices([set(), {1, 2}], n_samples=4))  # Sample 1 at locus 1
print(f"Different loci allowed: {cs_ok}")

## 8. Auto-updating After Coalescence

The constraint set can automatically remove constraints when their ancestral haplotype exists:

In [ ]:
cs = ARGConstraintSet(n_loci=2)
cs.add(ARGConstraint.from_indices([{0, 1}, {0, 1}], n_samples=4))
cs.add(ARGConstraint.from_indices([{2, 3}, {2, 3}], n_samples=4))
print(f"Initial: {len(cs)} constraints")

# Create lineages
lins = [ARGLineage.from_sample(i, 4, 2) for i in range(4)]

# Merge 0+1 (satisfies first constraint)
merged_01 = ARGLineage.coalesce(lins[0], lins[1])
lineages = {merged_01, lins[2], lins[3]}
cs.update_after_coalescence(lineages)
print(f"After 0+1: {len(cs)} constraints")

# Merge 2+3 (satisfies second constraint)
merged_23 = ARGLineage.coalesce(lins[2], lins[3])
lineages = {merged_01, merged_23}
cs.update_after_coalescence(lineages)
print(f"After 2+3: {len(cs)} constraints")

## 9. Efficient Pair Finding

The constraint set provides efficient methods to find compatible pairs without O(n²) pairwise checks:

In [ ]:
cs = ARGConstraintSet(n_loci=2)
cs.add(ARGConstraint.from_indices([{0, 1, 2}, {0, 1}], n_samples=6))
cs.add(ARGConstraint.from_indices([{3, 4}, {3, 4, 5}], n_samples=6))

lineages = {ARGLineage.from_sample(i, 6, 2) for i in range(6)}

# Efficient grouping - lineages in same group can coalesce
print("Compatible groups:")
groups = cs.get_compatible_groups(lineages)
for i, group in enumerate(groups):
    print(f"  Group {i}: {[str(l) for l in group]}")

print("\nCompatible pairs:")
pairs = cs.get_compatible_pairs(lineages)
for a, b in pairs:
    print(f"  {a} + {b}")

## 10. Complete Simulation Example

Here's a complete example simulating ARG coalescence with constraints, using efficient pair finding:

In [ ]:
import random

def simulate_arg_coalescence(n_samples, n_loci, constraints_spec, seed=42):
    """
    Simulate ARG coalescence with constraints.

    Args:
        n_samples: Number of samples
        n_loci: Number of loci
        constraints_spec: List of (indices_per_locus) tuples
        seed: Random seed
    """
    random.seed(seed)

    # Set up constraints
    cs = ARGConstraintSet(n_loci)
    for indices_per_locus in constraints_spec:
        cs.add(ARGConstraint.from_indices(indices_per_locus, n_samples))

    # Initialize lineages
    lineages = {ARGLineage.from_sample(i, n_samples, n_loci) for i in range(n_samples)}

    print(f"Samples: {n_samples}, Loci: {n_loci}")
    print(f"Constraints: {[list(c.indices_at_locus(l) for l in range(n_loci)) for c in cs]}")
    print()

    step = 0
    while len(lineages) > 1:
        # Use efficient pair finding
        compatible = cs.get_compatible_pairs(lineages)

        if not compatible:
            print(f"Step {step}: No compatible pairs - deadlock!")
            break

        # Randomly choose a pair
        lin_a, lin_b = random.choice(compatible)
        merged = ARGLineage.coalesce(lin_a, lin_b)

        # Update lineages
        lineages.remove(lin_a)
        lineages.remove(lin_b)
        lineages.add(merged)

        # Update constraints
        old_count = len(cs)
        cs.update_after_coalescence(lineages)
        satisfied = old_count - len(cs)

        print(f"Step {step}: {lin_a} + {lin_b}")
        print(f"       -> {merged}")
        if satisfied:
            print(f"       Satisfied {satisfied} constraint(s), {len(cs)} remaining")
        step += 1

    print(f"\nCoalescence complete in {step} steps")
    return lineages

In [ ]:
# Example: 6 samples, 2 loci, with two ancestral haplotype constraints
simulate_arg_coalescence(
    n_samples=6,
    n_loci=2,
    constraints_spec=[
        [{0, 1, 2}, {0, 1}],  # Haplotype must be ancestral to {0,1,2} at locus 0, {0,1} at locus 1
        [{3, 4}, {3, 4, 5}],  # Haplotype must be ancestral to {3,4} at locus 0, {3,4,5} at locus 1
    ]
)

## 11. Comparison: Tree vs ARG Constraints

| Feature | CladeConstraint (Tree) | ARGConstraint |
|---------|------------------------|---------------|
| Array shape | 1D `(n_samples,)` | 2D `(n_samples, n_loci)` |
| Lineage tracking | Single bitarray | Tuple of bitarrays (per locus) |
| Satisfaction | All samples in one lineage | All samples at all loci in one lineage |
| Coalescence | All-or-nothing | Can be locus-specific |